In [ ]:

!pip install opencv-python-headless

import cv2
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Image, clear_output
from google.colab.output import eval_js
from PIL import Image as PILImage
import io
import base64
from tensorflow.keras.models import load_model

In [ ]:

from google.colab import files

print("Please upload your trained MNIST model file (e.g., best_mnist_cnn_model.keras or mnist_model.keras)")
uploaded = files.upload()

model_filename = list(uploaded.keys())[0]
print(f"Uploaded model: {model_filename}")


model = load_model(model_filename)
print("Model loaded successfully!")

In [ ]:


def preprocess_for_mnist(image_bgr):
    """
    Takes a BGR image (from webcam), extracts the digit region,
    and returns a model-ready input (1, 28, 28, 1) or None if no digit found.
    """

    gray = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY)


    blurred = cv2.GaussianBlur(gray, (5, 5), 0)
    thresh = cv2.adaptiveThreshold(
        blurred, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY_INV, 11, 2
    )


    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    if not contours:
        return None, "No contours found"


    largest_contour = max(contours, key=cv2.contourArea)
    x, y, w, h = cv2.boundingRect(largest_contour)


    if w * h < 400:
        return None, "Digit too small"

    digit_roi = thresh[y:y+h, x:x+w]
    resized = cv2.resize(digit_roi, (28, 28))


    model_input = resized.astype("float32") / 255.0
    model_input = model_input.reshape(1, 28, 28, 1)

    return model_input, (x, y, w, h)

In [ ]:


def take_photo(width=640, height=480):
    """Capture a single photo from the webcam and return base64 data URL."""
    js_code = f"""
    async function takePhoto() {{
      const div = document.createElement('div');
      const video = document.createElement('video');
      video.style.display = 'block';
      const stream = await navigator.mediaDevices.getUserMedia({{video: true}});

      document.body.appendChild(div);
      div.appendChild(video);
      video.srcObject = stream;
      await video.play();

      // Resize the output to fit the video element.
      google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);

      // Wait for video to be ready
      await new Promise((resolve) => video.onloadedmetadata = resolve);

      const canvas = document.createElement('canvas');
      canvas.width = {width};
      canvas.height = {height};
      canvas.getContext('2d').drawImage(video, 0, 0, canvas.width, canvas.height);

      stream.getTracks().forEach(track => track.stop());
      div.remove();
      return canvas.toDataURL('image/png');
    }}
    """
    data_url = eval_js(js_code + 'takePhoto();')
    return data_url

In [ ]:


def predict_digit_from_webcam():
    print("📸 Taking photo... Please draw a large, clear digit and hold it in front of the camera.")
    print("   (Make sure there's good lighting and a plain background)")


    data_url = take_photo()


    header, encoded = data_url.split(",", 1)
    img_bytes = base64.b64decode(encoded)
    pil_img = PILImage.open(io.BytesIO(img_bytes))

    opencv_img = cv2.cvtColor(np.array(pil_img), cv2.COLOR_RGB2BGR)


    clear_output(wait=True)
    plt.figure(figsize=(8, 6))
    plt.imshow(cv2.cvtColor(opencv_img, cv2.COLOR_BGR2RGB))
    plt.title("Captured Image")
    plt.axis("off")
    plt.show()

    model_input, info = preprocess_for_mnist(opencv_img)

    if model_input is None:
        print(f"❌ Could not detect a digit: {info}")
        print("   Try drawing a bigger, bolder digit with more contrast.")
        return

    prediction = model.predict(model_input, verbose=0)
    predicted_digit = np.argmax(prediction)
    confidence = np.max(prediction) * 100

    print(f"✅ Predicted Digit: {predicted_digit}")
    print(f"   Confidence: {confidence:.1f}%")


    plt.figure(figsize=(10, 3))
    plt.bar(range(10), prediction[0])
    plt.xticks(range(10))
    plt.title("Prediction Confidence for Each Digit")
    plt.xlabel("Digit")
    plt.ylabel("Probability")
    plt.show()

    processed_28x28 = model_input[0, :, :, 0]
    plt.figure(figsize=(4, 4))
    plt.imshow(processed_28x28, cmap='gray')
    plt.title(f"Processed Input to Model\n(Predicted: {predicted_digit})")
    plt.axis("off")
    plt.show()


predict_digit_from_webcam()

In [ ]:

predict_digit_from_webcam()